In [1]:
"""
Smoke test: load a HF-format model and run a short prediction on a synthetic sentence.
"""
import pathlib
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification


def load_model(model_dir: str):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    return model, tokenizer, device


def predict_words(model, tokenizer, device, words: list):
    enc = tokenizer(
        words, is_split_into_words=True, return_tensors="pt", truncation=True
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        out = model(**enc)
        logits = out.logits.cpu().numpy()[0]

    word_ids = tokenizer(words, is_split_into_words=True)["input_ids"]
    # Use model.config.id2label when available
    cfg_id2label = getattr(model.config, "id2label", None)
    if cfg_id2label:
        id2label = {int(k): v for k, v in cfg_id2label.items()}
    else:
        id2label = None

    # Map per-token preds to first-subtoken-per-word labels
    tokens_word_ids = tokenizer(words, is_split_into_words=True).word_ids()
    preds = logits.argmax(axis=-1)
    word_preds = []
    previous_word = None
    for token_pred, widx in zip(preds, tokens_word_ids):
        if widx is None:
            previous_word = None
            continue
        if widx != previous_word:
            label = (
                id2label.get(int(token_pred), str(int(token_pred)))
                if id2label
                else str(int(token_pred))
            )
            word_preds.append(label)
        previous_word = widx

    return word_preds

In [ ]:
model_dir = pathlib.Path("../models/NER_mudel_v2/")
model, tokenizer, device = load_model(model_dir)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
sentence = "See on test lause"
words = sentence.split()
preds = predict_words(model, tokenizer, device, words)
print("Words:", words)
print("Predicted labels:", preds)

Words: ['See', 'on', 'test', 'lause']
Predicted labels: ['?', 'b_V', 'sg n_A', 'sg n_S']
